# nb_mdf_56_compare_paths - parallel run: notebook path vs materialized lake view path

The same **migrate -> validate -> run alongside -> cut over** pattern most teams use when moving from SSIS/SQL to Fabric
applies inside Fabric when you change *how* a layer is built. Both paths read the same Bronze and the same
metadata, so their Silver and Gold should agree. This notebook proves it, table by table:

* row counts (Silver compares current, non-deleted rows; SCD2 also compares version counts)
* a two-way `EXCEPT ALL` on the business columns of every Gold table (audit columns excluded)

Typical, explainable differences (worth discussing with the class):
* **Rule evaluated on the latest version only** - the notebook keeps the last *good* version when an update
  is quarantined; the MLV evaluates the newest version and drops the key until the source is fixed.
* **Gold using `current_date()`** - compare on the same day, or the age-based columns differ.
* **`_ingested_at`-based columns** - identical only when both paths read the same Bronze batches.

In [ ]:
write_results = True

In [ ]:
%run nb_mdf_utils

In [ ]:
defs = spark.table(tname("meta", "mlv_definition")).where("status = 'CREATED'")
objs = {o["silver_table"]: o for o in get_objects()}
rows = []

def business_cols(a, b):
    """Columns both sides share, excluding framework/audit columns (they legitimately differ)."""
    ca = {c.lower(): c for c in spark.table(a).columns}
    cb = {c.lower() for c in spark.table(b).columns}
    return [ca[c] for c in ca if c in cb and not c.startswith("_")]


def current_rows(t):
    """Current, non-deleted rows whatever lifecycle columns the table carries."""
    cols = {c.lower() for c in spark.table(t).columns}
    conds = (["_is_current = true"] if "_is_current" in cols else []) + \
            (["coalesce(_is_deleted, false) = false"] if "_is_deleted" in cols else [])
    return spark.table(t).where(" AND ".join(conds) or "true")

for d in defs.where("layer = 'silver'").collect():
    ent = d["mlv_name"].split(".")[-1]
    nb_t = tname("silver", ent)
    if not table_exists(nb_t):
        rows.append(("silver", ent, None, None, None, None, "NOTEBOOK TABLE MISSING")); continue
    cols = business_cols(nb_t, d["mlv_name"])
    a, b = current_rows(nb_t).select(*cols), current_rows(d["mlv_name"]).select(*cols)
    n_nb, n_mlv = a.count(), b.count()
    only_nb, only_mlv = a.exceptAll(b).count(), b.exceptAll(a).count()
    rows.append(("silver", ent, n_nb, n_mlv, only_nb, only_mlv, "MATCH" if only_nb == only_mlv == 0 else "DIFF"))

for d in defs.where("layer = 'gold'").collect():
    tbl = d["mlv_name"].split(".")[-1]
    nb_t = tname("gold", tbl)
    if not table_exists(nb_t):
        rows.append(("gold", tbl, None, None, None, None, "NOTEBOOK TABLE MISSING")); continue
    cols = business_cols(nb_t, d["mlv_name"])
    a, b = spark.table(nb_t).select(*cols), spark.table(d["mlv_name"]).select(*cols)
    only_nb, only_mlv = a.exceptAll(b).count(), b.exceptAll(a).count()
    rows.append(("gold", tbl, a.count(), b.count(), only_nb, only_mlv, "MATCH" if only_nb == only_mlv == 0 else "DIFF"))

res = spark.createDataFrame(rows, "layer string, entity string, notebook_rows long, mlv_rows long, "
                                  "only_in_notebook long, only_in_mlv long, verdict string") \
           .withColumn("compared_at", F.current_timestamp())
display(res.orderBy("layer", "verdict", "entity"))
if write_results:
    res.write.mode("append").option("mergeSchema", "true").saveAsTable(tname("meta", "parallel_run_check"))

## Drill into a difference
Set `entity` to a row marked DIFF and look at the keys on each side.

In [ ]:
entity = next((r[1] for r in rows if r[6] == "DIFF"), None)
if entity:
    layer = next(r[0] for r in rows if r[1] == entity)
    nb_t, mlv_t = tname(layer, entity), f"{layer}_mlv.{entity}"
    cols = business_cols(nb_t, mlv_t)
    print("Only in notebook path:"); display(spark.table(nb_t).select(*cols).exceptAll(spark.table(mlv_t).select(*cols)).limit(20))
    print("Only in MLV path:"); display(spark.table(mlv_t).select(*cols).exceptAll(spark.table(nb_t).select(*cols)).limit(20))
else:
    print("Every compared table matches.")